# 01. 인프라 배포 및 AI Search 파이프라인 설정

이 노트북에서는 Bicep으로 전체 인프라를 배포하고, Azure AI Search 파이프라인(인덱서 + 스킬셋)을 설정합니다.

## 배포 리전
**Sweden Central** — Document Intelligence가 Korea Central에서 미지원이므로 Sweden Central 사용

## 배포 리소스
| 리소스 | SKU | 용도 |
|--------|-----|------|
| Virtual Network (10.0.0.0/16) | - | Private Network 격리 |
| Storage Account | Standard LRS | 크롤링 데이터 저장 (Private) |
| Azure AI Services | S0 | GPT-5.4, text-embedding-3-large |
| Document Intelligence | S0 | PDF 레이아웃 분석 |
| Azure AI Search | **Standard (S1)** | 벡터 + 시맨틱 하이브리드 검색 |
| Function App (EP1) | Elastic Premium | Python 크롤러 (VNet Integration) |
| Logic App | Consumption | 크롤 스케줄러 (매일 21:00 UTC) |
| Private Endpoints × 4 | - | 모든 서비스 프라이빗 접근 |
| Shared Private Links × 2 | - | AI Search 아웃바운드 |

## 노트북 흐름
```
1. 사전 요구사항 확인
2. Bicep 배포 (swedencentral, Private Network)
3. Function App 코드 배포 (crawl-function/)
4. Shared Private Link 승인 (Storage, AI Services)
5. AI Search 파이프라인 설정 (Index + Skillset + Indexer)
6. .env 파일 생성
7. 배포 검증
```

## 1. 사전 요구사항 확인

In [1]:
# Azure CLI 로그인 확인
!az account show --output table

EnvironmentName    HomeTenantId                          IsDefault    Name                            State    TenantDefaultDomain               TenantDisplayName    TenantId
-----------------  ------------------------------------  -----------  ------------------------------  -------  --------------------------------  -------------------  ------------------------------------
AzureCloud         ebed7cce-48ed-45fe-ad61-fddfc8e2fb54  True         ME-MngEnvMCAP719772-jihyeseo-1  Enabled  MngEnvMCAP719772.onmicrosoft.com  Contoso              ebed7cce-48ed-45fe-ad61-fddfc8e2fb54


In [ ]:
from getpass import getpass

# JumpVM 로컬 관리자 비밀번호를 런타임에 안전하게 입력
JUMPVM_ADMIN_PASSWORD = getpass("JumpVM admin password: ")

# JumpVM Entra 로그인 허용 대상
JUMPVM_ENTRA_USER_OBJECT_IDS = "['4cb458b5-ef0c-403d-8162-c39855018986']"

In [2]:
import subprocess
import json

# 구독 ID 설정 (az account show에서 확인)
result = subprocess.run(
    ["az", "account", "show", "--query", "id", "--output", "tsv"],
    capture_output=True, text=True
)
SUBSCRIPTION_ID = result.stdout.strip()
RG_NAME = "rg-rag-indexing-lab-swc"
LOCATION = "swedencentral"
DEPLOYMENT_NAME = "rag-indexing-lab-swc"

print(f"Subscription: {SUBSCRIPTION_ID}")
print(f"Resource Group: {RG_NAME}")
print(f"Location: {LOCATION}")

Subscription: 8fcfff7e-0a34-491c-a1e5-23b22c1736de
Resource Group: rg-rag-indexing-lab-swc
Location: swedencentral


In [3]:
# 필수 도구 확인
!az bicep version
!func --version  # Azure Functions Core Tools

==Bicep CLI version 0.43.8 (310735909d)

=4.10.0


## 2. Bicep 배포

Private Network(VNet + Private Endpoints) 전체 인프라를 한 번에 배포합니다.  
약 10~15분 소요됩니다.

현재 JumpVM 표준 이름은 `jumpvmragi01`이며, Entra 로그인과 Windows 11 Pro 이미지 기준으로 재배포됩니다.
`jumpvmAdminPassword`는 secure 파라미터이므로 아래 셀에서 런타임에 입력합니다.

In [ ]:
# Sweden Central에 배포 (~10분 소요)
!az deployment sub create \
    --location swedencentral \
    --template-file ../infra/main.bicep \
    --name {DEPLOYMENT_NAME} \
    --parameters resourceGroupName={RG_NAME} \
    --parameters location={LOCATION} \
    --parameters embeddingDeploymentName='text-embedding-3-large' \
    --parameters gpt54DeploymentName='gpt-5.4' \
    --parameters gpt54ModelVersion='2026-03-05' \
    --parameters searchSku='standard' \
    --parameters blobContainerName='raw-documents' \
    --parameters jumpvmAdminUsername='azureadmin' \
    --parameters jumpvmAdminPassword="{JUMPVM_ADMIN_PASSWORD}" \
    --parameters jumpvmEntraUserObjectIds="{JUMPVM_ENTRA_USER_OBJECT_IDS}" \
    --output table

Bicep CLI is already installed at '/home/azureuser/.azure/bin/bicep'. Skipping installation as no specific version was requested.
===/home/azureuser/localfiles/azure-rag-indexing-lab/infra/modules/vnet.bicep(69,27) : Warning no-hardcoded-env-urls: Environment URLs should not be hardcoded. Use the environment() function to ensure compatibility across clouds. Found this disallowed host: "core.windows.net" [https://aka.ms/bicep/linter-diagnostics#no-hardcoded-env-urls]

Name                  State      Timestamp                         Mode
--------------------  ---------  --------------------------------  -----------
rag-indexing-lab-swc  Succeeded  2026-05-11T08:45:44.710753+00:00  Incremental


In [6]:
# 배포 출력값 파싱
result = subprocess.run(
    ["az", "deployment", "sub", "show",
     "--name", DEPLOYMENT_NAME,
     "--query", "properties.outputs",
     "--output", "json"],
    capture_output=True, text=True
)
outputs = json.loads(result.stdout)

print("=== 배포된 리소스 엔드포인트 ===")
for key, val in outputs.items():
    print(f"  {key}: {val['value']}")

=== 배포된 리소스 엔드포인트 ===
  aiSearchEndpoint: https://search-ragi-dyn6dtfu.search.windows.net
  aiSearchName: search-ragi-dyn6dtfu
  aiSearchPrincipalId: 55954e38-f9f2-4ce9-a274-a5f39dcddc37
  crawlFunctionName: func-crawl-ragi-dyn6dtfu
  crawlFunctionUrl: https://func-crawl-ragi-dyn6dtfu.azurewebsites.net/api/crawl
  crawlLogicAppName: logic-crawl-ragi-dyn6dtfu
  docIntelligenceEndpoint: https://di-ragi-dyn6dtfu.cognitiveservices.azure.com/
  gpt54DeploymentName: gpt-5.4
  location: swedencentral
  openaiAccountName: ais-ragi-dyn6dtfu
  openaiEndpoint: https://ais-ragi-dyn6dtfu.cognitiveservices.azure.com/
  resourceGroupName: rg-rag-indexing-lab-swc
  storageAccountBlobEndpoint: https://stragidyn6dtfun6.blob.core.windows.net/
  storageAccountName: stragidyn6dtfun6
  vnetName: vnet-ragi-dyn6dtfu


## 3. Function App 코드 배포

Bicep으로 인프라만 생성했으므로, 크롤러 코드(`crawl-function/`)를 Function App에 배포합니다.

In [7]:
FUNC_APP_NAME = outputs['crawlFunctionName']['value']
print(f"Function App: {FUNC_APP_NAME}")

# Azure Functions Core Tools로 배포
import subprocess
result = subprocess.run(
    ["func", "azure", "functionapp", "publish", FUNC_APP_NAME, "--python"],
    capture_output=True, text=True,
    cwd="../crawl-function"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

Function App: func-crawl-ragi-dyn6dtfu
Local python version '3.10.11' is different from the version expected for your deployed Function App. This may result in 'ModuleNotFound' errors in Azure Functions. Please create a Python Function App for version 3.10 or change the virtual environment on your local machine to match 'Python|3.11'.
Getting site publishing info...
[2026-05-11T08:46:35.340Z] Starting the function app deployment...
Updating Application Settings for Remote build...
Creating archive for current directory...
Performing remote build for functions project.
Deleting the old .python_packages directory
Uploading 9.94 KB [----------------------------------------------------------Remote build in progress, please wait...
Updating submodules.
Preparing deployment for commit id '6d0b0f74-0'.
PreDeployment: context.CleanOutputPath False
PreDeployment: context.OutputPath /home/site/wwwroot
Repository path is /tmp/zipdeploy/extracted
Running oryx build...
Command: oryx build /tmp/zipd

In [8]:
import subprocess, json, requests as req

# az functionapp function list (Python v2 모델은 런타임 기동 후 등록됨)
result = subprocess.run(
    ["az", "functionapp", "function", "list",
     "--name", FUNC_APP_NAME,
     "--resource-group", RG_NAME,
     "--output", "json"],
    capture_output=True, text=True
)
funcs = json.loads(result.stdout) if result.stdout.strip() else []

if funcs:
    for f in funcs:
        short_name = f["name"].split("/")[-1]   # "func-app/crawl" → "crawl"
        print(f"  ✅ {short_name}  →  {f.get('invokeUrlTemplate', '')}")
else:
    # Python v2는 cold start 전 CLI에서 빈 결과 반환 — HTTP로 직접 확인
    print("CLI 조회 결과 없음 (Python v2 cold start 전 정상). HTTP 엔드포인트 확인 중...")
    crawl_url = CRAWL_FUNC_URL if 'CRAWL_FUNC_URL' in dir() else f"https://{FUNC_APP_NAME}.azurewebsites.net/api/crawl"
    try:
        r = req.get(crawl_url, timeout=30)
        print(f"  HTTP {r.status_code} — Function 응답 확인 ({'정상' if r.status_code < 500 else '오류'})")
        print(f"  URL: {crawl_url}")
    except Exception as e:
        print(f"  연결 실패: {e}")


  ✅ crawl  →  https://func-crawl-ragi-dyn6dtfu.azurewebsites.net/api/crawl


## 4. Shared Private Link 승인

AI Search가 아웃바운드 Shared Private Link를 통해 Storage, AI Services, Document Intelligence에 접근하려면  
각 리소스에서 연결 요청을 **수동으로 승인**해야 합니다.

> Bicep 배포 후 약 5~10분 후 승인 가능

In [9]:
SEARCH_NAME = outputs['aiSearchName']['value']
print(f"AI Search: {SEARCH_NAME}")

# Shared Private Link 상태 확인
!az search shared-private-link-resource list \
    --service-name {SEARCH_NAME} \
    --resource-group {RG_NAME} \
    --query "[].{{name:name, status:properties.status}}" \
    --output table

AI Search: search-ragi-dyn6dtfu
Name      Status
--------  --------
spl-blob  Approved


In [10]:
STORAGE_NAME = outputs['storageAccountName']['value']

# Storage Account PE 연결 목록 확인
result = subprocess.run(
    ["az", "network", "private-endpoint-connection", "list",
     "--name", STORAGE_NAME,
     "--resource-group", RG_NAME,
     "--type", "Microsoft.Storage/storageAccounts",
     "--query", "[?properties.privateLinkServiceConnectionState.status=='Pending'].name",
     "--output", "tsv"],
    capture_output=True, text=True
)
pending = [c for c in result.stdout.strip().split('\n') if c]

for conn in pending:
    print(f"승인: {conn}")
    !az network private-endpoint-connection approve \
        --name {conn} \
        --resource-name {STORAGE_NAME} \
        --resource-group {RG_NAME} \
        --type Microsoft.Storage/storageAccounts \
        --description "Approved for AI Search indexer" \
        --output none

if not pending:
    print("승인 대기 중인 Storage 연결 없음 (이미 승인됨 또는 아직 생성 중)")

승인 대기 중인 Storage 연결 없음 (이미 승인됨 또는 아직 생성 중)


In [11]:
AI_SERVICES_NAME = outputs['openaiAccountName']['value']
DI_NAME = f"di-ragi-{STORAGE_NAME[6:14]}"  # suffix 추출

for resource_name in [AI_SERVICES_NAME, DI_NAME]:
    result = subprocess.run(
        ["az", "network", "private-endpoint-connection", "list",
         "--name", resource_name,
         "--resource-group", RG_NAME,
         "--type", "Microsoft.CognitiveServices/accounts",
         "--query", "[?properties.privateLinkServiceConnectionState.status=='Pending'].name",
         "--output", "tsv"],
        capture_output=True, text=True
    )
    for conn in [c for c in result.stdout.strip().split('\n') if c]:
        print(f"승인: {resource_name} / {conn}")
        !az network private-endpoint-connection approve \
            --name {conn} \
            --resource-name {resource_name} \
            --resource-group {RG_NAME} \
            --type Microsoft.CognitiveServices/accounts \
            --description "Approved for AI Search skillset" \
            --output none

In [12]:
# 최종 승인 상태 확인 — 모두 Approved 여야 AI Search 파이프라인 작동
!az search shared-private-link-resource list \
    --service-name {SEARCH_NAME} \
    --resource-group {RG_NAME} \
    --query "[].{{name:name, status:properties.status, groupId:properties.groupId}}" \
    --output table

Name      Status    GroupId
--------  --------  ---------
spl-blob  Approved  blob


## 5. AI Search 파이프라인 설정

AI Search 네이티브 Indexer + Skillset을 생성합니다:
- **Data Source**: `raw-documents` Blob 컨테이너 감시
- **Skillset**: SplitSkill → AzureOpenAIEmbeddingSkill
- **Index**: `law-documents-index` (벡터 3072D + 시맨틱)
- **Indexer**: 매일 06:00 UTC 자동 실행

> **Note**: AI Search가 Private Network 내에 있으므로, 임시로 Public Access를 허용하거나 VPN/Bastion 경유가 필요합니다.

In [ ]:
# AI Search 임시 Public Access 허용 (스크립트 실행 후 다시 비활성화)
!az search service update \
    --name {SEARCH_NAME} \
    --resource-group {RG_NAME} \
    --public-network-access enabled \
    --output none

print("AI Search public access 임시 허용")

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

SEARCH_ENDPOINT = outputs['aiSearchEndpoint']['value']
OPENAI_ENDPOINT = outputs['openaiEndpoint']['value']
OPENAI_ACCOUNT  = outputs['openaiAccountName']['value']
STORAGE_ACCOUNT = outputs['storageAccountName']['value']

# 환경변수 세팅 후 파이프라인 설정 스크립트 실행
env = {
    **os.environ,
    "AZURE_SEARCH_SERVICE_ENDPOINT": SEARCH_ENDPOINT,
    "AZURE_OPENAI_ENDPOINT": OPENAI_ENDPOINT,
    "AZURE_OPENAI_ACCOUNT_NAME": OPENAI_ACCOUNT,
    "AZURE_STORAGE_ACCOUNT_NAME": STORAGE_ACCOUNT,
    "AZURE_RESOURCE_GROUP": RG_NAME,
}

result = subprocess.run(
    [sys.executable, "../scripts/setup_ai_search_pipeline.py"],
    capture_output=True, text=True, env=env
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

In [ ]:
# 인덱서 즉시 실행 (초기 인덱싱)
result = subprocess.run(
    [sys.executable, "../scripts/setup_ai_search_pipeline.py", "--run"],
    capture_output=True, text=True, env=env
)
print(result.stdout)

In [ ]:
# AI Search Public Access 다시 비활성화
!az search service update \
    --name {SEARCH_NAME} \
    --resource-group {RG_NAME} \
    --public-network-access disabled \
    --output none

print("AI Search public access 비활성화 완료")

## 6. .env 파일 생성

배포 출력값으로 `.env` 파일을 자동 생성합니다.  
모든 인증은 **Managed Identity (DefaultAzureCredential)** 기반으로, API 키가 필요 없습니다.

In [ ]:
CRAWL_FUNC_URL  = outputs.get('crawlFunctionUrl', {}).get('value', '')
LOGIC_APP_NAME  = outputs.get('crawlLogicAppName', {}).get('value', '')
GPT54_DEPLOY    = outputs.get('gpt54DeploymentName', {}).get('value', 'gpt-5.4')
DI_ENDPOINT     = outputs['docIntelligenceEndpoint']['value']
STORAGE_RESOURCE_ID = (
    f"/subscriptions/{SUBSCRIPTION_ID}"
    f"/resourceGroups/{RG_NAME}"
    f"/providers/Microsoft.Storage/storageAccounts/{STORAGE_ACCOUNT}"
)

env_content = f"""# Auto-generated from Bicep deployment — DO NOT commit to git
AZURE_SUBSCRIPTION_ID={SUBSCRIPTION_ID}
AZURE_RESOURCE_GROUP={RG_NAME}
AZURE_LOCATION=swedencentral

# Storage (Private — Managed Identity 인증)
AZURE_STORAGE_ACCOUNT_NAME={STORAGE_ACCOUNT}
AZURE_STORAGE_CONTAINER_NAME=raw-documents
AZURE_STORAGE_RESOURCE_ID={STORAGE_RESOURCE_ID}
BLOB_CONTAINER_NAME=raw-documents

# Azure AI Services (OpenAI)
AZURE_OPENAI_ENDPOINT={OPENAI_ENDPOINT}
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=text-embedding-3-large
AZURE_OPENAI_GPT54_DEPLOYMENT={GPT54_DEPLOY}

# Azure AI Search
AZURE_SEARCH_SERVICE_ENDPOINT={SEARCH_ENDPOINT}
AZURE_SEARCH_INDEX_NAME=law-documents-index

# Document Intelligence
AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT={DI_ENDPOINT}

# Crawl Pipeline
AZURE_FUNCTION_CRAWL_URL={CRAWL_FUNC_URL}
AZURE_LOGIC_APP_NAME={LOGIC_APP_NAME}
"""

with open('../.env', 'w') as f:
    f.write(env_content)

print(".env 파일 생성 완료 (Managed Identity 기준 변수 포함)")
print(env_content)

## 7. 배포 검증

In [ ]:
# 리소스 그룹 내 전체 리소스 확인
!az resource list --resource-group {RG_NAME} --output table

In [ ]:
import requests

# Function App 수동 크롤 테스트 (limit=2로 빠르게)
resp = requests.post(
    CRAWL_FUNC_URL,
    json={"limit": 2, "triggered_by": "notebook-test"},
    timeout=120
)
print(f"Status: {resp.status_code}")
print(resp.json())

In [ ]:
# AI Search 인덱서 상태 확인
import time
from azure.identity import DefaultAzureCredential
import requests as req

credential = DefaultAzureCredential()
token = credential.get_token("https://search.azure.com/.default").token

# 인덱서 상태 조회
r = req.get(
    f"{SEARCH_ENDPOINT}/indexers/law-blob-indexer/status?api-version=2024-07-01",
    headers={"Authorization": f"Bearer {token}"}
)
status = r.json()
last_run = status.get("lastResult", {})
print(f"인덱서 상태: {last_run.get('status', 'N/A')}")
print(f"처리된 문서: {last_run.get('itemsProcessed', 0)}")
print(f"실패한 문서: {last_run.get('itemsFailed', 0)}")
print(f"완료 시각: {last_run.get('endTime', 'N/A')}")

---
다음 단계: [02-data-crawling.ipynb](02-data-crawling.ipynb) — Azure Function App으로 크롤링 실행 및 모니터링